# Buổi 5: Xử lý và biến đổi dữ liệu với Pandas
**Môn học:** Python cho Phân tích Dữ liệu Kinh doanh · **BÀI TẬP THỰC HÀNH**

**Mục tiêu:**
- Phát hiện và xử lý dữ liệu trùng lặp, thiếu, sai định dạng, ngoại lai
- Chuyển kiểu dữ liệu, xử lý ngày tháng, tạo cột mới
- Tổng hợp bằng groupby, pivot_table; kết hợp bảng bằng merge, concat
- Tính chỉ số RFM cho khách hàng

**Dữ liệu dùng trong buổi:** `ecommerce_orders_raw.csv`, `customers.csv` (trong thư mục `data/`). Nếu dùng Google Colab, tải thư mục `data/` lên rồi sửa biến `DATA` bên dưới.

**Cấu trúc:** Phần A – bài tập có hướng dẫn (làm theo, chạy từng ô) · Phần B – bài tập tự làm (bắt buộc, nộp cuối buổi) · Phần C – bài tập làm thêm (dành cho bạn muốn thử thách).

In [ ]:
import numpy as np
import pandas as pd
DATA = "../data/"
pd.set_option("display.max_columns", 30)

## Phần A. Bài tập có hướng dẫn

Làm sạch bộ dữ liệu đơn hàng thương mại điện tử năm 2025 rồi phân tích.

### A1. Khám phá vấn đề của dữ liệu thô

In [ ]:
raw = pd.read_csv(DATA + "ecommerce_orders_raw.csv")
print(raw.shape)
raw.info()
raw.sample(8, random_state=3)

In [ ]:
print("Giá trị thiếu mỗi cột:\n", raw.isna().sum())
print("\nSố dòng trùng hoàn toàn:", raw.duplicated().sum())
print("\nCác giá trị cột vùng:", raw["vung"].unique())
print("\nVí dụ đơn giá:", raw["don_gia"].sample(6, random_state=1).tolist())

**Nhận xét:** (1) có dòng trùng; (2) `vung` viết không thống nhất và có giá trị thiếu; (3) `so_luong` thiếu;
(4) `don_gia` là chuỗi, có dạng "1.250.000 đ", có giá trị âm và giá trị bất thường; (5) `ngay_dat` lẫn hai định dạng.

### A2. Bỏ trùng lặp

In [ ]:
df = raw.drop_duplicates().copy()
print("Trước:", len(raw), "→ Sau:", len(df))
print("Mã đơn còn trùng?", df["ma_don"].duplicated().sum())

### A3. Chuẩn hóa chuỗi và kiểu số

In [ ]:
df["vung"] = df["vung"].str.strip().str.title()
print(df["vung"].value_counts(dropna=False))

def lam_sach_gia(x):
    if isinstance(x, str):
        x = x.replace("đ", "").replace(".", "").strip()
    return float(x)

df["don_gia"] = df["don_gia"].apply(lam_sach_gia)
df["so_luong"] = pd.to_numeric(df["so_luong"], errors="coerce")
df[["don_gia", "so_luong"]].describe()

### A4. Ngày tháng
`pd.to_datetime` với `format="mixed"` và `dayfirst=True` hiểu được cả `2025-03-15` lẫn `15/03/2025`.
Sau đó trích xuất tháng, quý, thứ trong tuần bằng `.dt`.

In [ ]:
df["ngay_dat"] = pd.to_datetime(df["ngay_dat"], format="mixed", dayfirst=True)
df["thang"] = df["ngay_dat"].dt.month
df["quy"] = df["ngay_dat"].dt.quarter
df["thu"] = df["ngay_dat"].dt.day_name()
print(df["ngay_dat"].min(), "→", df["ngay_dat"].max())
df[["ngay_dat", "thang", "quy", "thu"]].head()

### A5. Giá trị thiếu và ngoại lai
Chiến lược phổ biến: **xóa** dòng (khi ít và không quan trọng), **điền** giá trị (trung vị, mode, giá trị từ nguồn khác),
hoặc **giữ** và đánh dấu. Ngoại lai phát hiện bằng quy tắc IQR: ngoài khoảng `[Q1 − 1,5·IQR; Q3 + 1,5·IQR]`
tính **trong từng nhóm hàng** (vì laptop vốn đắt hơn phụ kiện).

In [ ]:
# 1) số lượng thiếu → điền bằng trung vị (1)
df["so_luong"] = df["so_luong"].fillna(df["so_luong"].median())

# 2) vùng thiếu → lấy từ bảng khách hàng
kh = pd.read_csv(DATA + "customers.csv")
vung_kh = kh.set_index("ma_kh")["vung"]
df["vung"] = df["vung"].fillna(df["ma_kh"].map(vung_kh))

# 3) đơn giá âm → lỗi nhập liệu, loại bỏ
df = df[df["don_gia"] > 0]

# 4) ngoại lai theo IQR trong từng nhóm hàng
def la_ngoai_lai(g):
    q1, q3 = g.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (g < q1 - 1.5 * iqr) | (g > q3 + 1.5 * iqr)

df["ngoai_lai"] = df.groupby("nhom_hang")["don_gia"].transform(la_ngoai_lai)
print("Số ngoại lai:", df["ngoai_lai"].sum())
print(df[df["ngoai_lai"]][["ma_don", "nhom_hang", "don_gia"]])
df = df[~df["ngoai_lai"]].drop(columns="ngoai_lai")
print("Còn lại:", len(df), "dòng; thiếu:", df.isna().sum().sum())

### A6. Cột mới, lọc đơn hoàn thành và tổng hợp

In [ ]:
df["so_luong"] = df["so_luong"].astype(int)
df["doanh_thu"] = df["so_luong"] * df["don_gia"]
ht = df[df["trang_thai"] == "Hoàn thành"].copy()
print("Tỷ lệ hủy đơn:", round((df["trang_thai"] == "Đã hủy").mean() * 100, 1), "%")

theo_thang = ht.groupby("thang").agg(so_don=("ma_don", "count"),
                                     doanh_thu_ty=("doanh_thu", lambda x: x.sum() / 1e9))
theo_thang.round(2)

`pivot_table` – bảng tổng hợp hai chiều giống PivotTable trong Excel:

In [ ]:
pv = pd.pivot_table(ht, index="nhom_hang", columns="kenh", values="doanh_thu",
                    aggfunc="sum", margins=True, margins_name="Tổng") / 1e6
pv.round(0)

`merge` ghép thông tin khách hàng (tuổi, giới tính) vào đơn hàng:

In [ ]:
ht = ht.merge(kh[["ma_kh", "tuoi", "gioi_tinh"]], on="ma_kh", how="left")
ht["nhom_tuoi"] = pd.cut(ht["tuoi"], bins=[17, 25, 35, 50, 65], labels=["18-25", "26-35", "36-50", "51-65"])
ht.groupby("nhom_tuoi", observed=True)["doanh_thu"].agg(["count", "mean"]).round(0)

### A7. Phân tích RFM
RFM đánh giá giá trị khách hàng bằng 3 chỉ số: **R**ecency – số ngày từ lần mua gần nhất,
**F**requency – số đơn, **M**onetary – tổng chi tiêu. Mỗi chỉ số chấm điểm 1–5 bằng ngũ phân vị (`pd.qcut`).

In [ ]:
ngay_moc = ht["ngay_dat"].max() + pd.Timedelta(days=1)
rfm = ht.groupby("ma_kh").agg(
    R=("ngay_dat", lambda x: (ngay_moc - x.max()).days),
    F=("ma_don", "nunique"),
    M=("doanh_thu", "sum"),
)
rfm["R_diem"] = pd.qcut(rfm["R"], 5, labels=[5, 4, 3, 2, 1]).astype(int)          # gần đây → điểm cao
rfm["F_diem"] = pd.qcut(rfm["F"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M_diem"] = pd.qcut(rfm["M"], 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["RFM"] = rfm["R_diem"].astype(str) + rfm["F_diem"].astype(str) + rfm["M_diem"].astype(str)
rfm.sort_values("M", ascending=False).head()

In [ ]:
def phan_khuc(r):
    if r.R_diem >= 4 and r.F_diem >= 4:
        return "Khách hàng tốt nhất"
    if r.F_diem >= 4:
        return "Trung thành"
    if r.R_diem >= 4:
        return "Mới / tiềm năng"
    if r.R_diem <= 2 and r.F_diem >= 3:
        return "Có nguy cơ rời bỏ"
    if r.R_diem <= 2:
        return "Ngủ đông"
    return "Cần chăm sóc"

rfm["phan_khuc"] = rfm.apply(phan_khuc, axis=1)
tong_hop = rfm.groupby("phan_khuc").agg(so_kh=("R", "size"), R_tb=("R", "mean"), F_tb=("F", "mean"), M_tb_trieu=("M", lambda x: x.mean() / 1e6))
tong_hop.round(1).sort_values("M_tb_trieu", ascending=False)

In [ ]:
ht.to_csv("don_hang_sach.csv", index=False, encoding="utf-8-sig")
rfm.to_csv("rfm.csv", encoding="utf-8-sig")

## Phần B. Bài tập tự làm

Dùng các DataFrame `df` (đã làm sạch, gồm cả đơn hủy), `ht` (đơn hoàn thành) và `rfm` ở Phần A.

### Bài B1. Báo cáo chất lượng dữ liệu

Viết hàm `kiem_tra(df)` trả về DataFrame có một dòng cho mỗi cột với: kiểu dữ liệu, số giá trị thiếu, % thiếu,
số giá trị khác nhau. Chạy hàm trên `raw` và trên `df` để so sánh trước – sau khi làm sạch.

In [ ]:
# Viết code của bạn ở đây


### Bài B2. Doanh thu theo vùng và quý

Từ `ht`, lập pivot table doanh thu (triệu đồng) với dòng là vùng, cột là quý. Thêm cột tăng trưởng quý 4 so với quý 3 (%).
Vùng nào tăng trưởng mạnh nhất?

In [ ]:
# Viết code của bạn ở đây


### Bài B3. Tỷ lệ hủy đơn theo kênh

Từ `df` (gồm cả đơn hủy), tính tỷ lệ hủy đơn (%) theo từng kênh bán và theo nhóm hàng.
Kênh/nhóm hàng nào có tỷ lệ hủy cao nhất? Gợi ý một hành động cho quản lý.

<details><summary>💡 Gợi ý</summary>

Tạo cột 0/1 cho đơn hủy rồi lấy trung bình theo nhóm = tỷ lệ.

</details>

In [ ]:
# Viết code của bạn ở đây


### Bài B4. Top sản phẩm và giỏ hàng trung bình

(a) 10 sản phẩm có doanh thu cao nhất và tỷ trọng cộng dồn (%) của chúng (gợi ý `cumsum`).
(b) Giá trị đơn hàng trung bình (AOV) theo kênh và theo tháng; tháng nào AOV cao nhất?

In [ ]:
# Viết code của bạn ở đây


### Bài B5. Nối dữ liệu RFM với thông tin khách hàng

Ghép `rfm` với `customers.csv` (dùng `merge`). Cho biết với mỗi phân khúc: số khách, tuổi trung bình, tỷ lệ nữ (%)
và vùng có nhiều khách nhất. Khách hàng có nguy cơ rời bỏ tập trung ở vùng nào?

In [ ]:
# Viết code của bạn ở đây


## Phần C. Bài tập làm thêm

### Bài C1. Phân tích nhóm khách hàng theo tháng mua đầu tiên (cohort)

Xác định tháng mua đầu tiên của mỗi khách hàng. Lập bảng cohort: dòng = tháng mua đầu, cột = số tháng kể từ lần đầu
(0, 1, 2, ...), giá trị = tỷ lệ khách còn quay lại mua (%). Nhận xét tỷ lệ giữ chân sau 3 tháng.

In [ ]:
# Viết code của bạn ở đây


### Bài C2. Doanh thu theo ngày trong tuần và trung bình trượt

(a) Doanh thu trung bình theo thứ trong tuần (sắp xếp Thứ 2 → Chủ nhật, đổi tên sang tiếng Việt).
(b) Tạo chuỗi doanh thu theo ngày (`resample("D")`), tính trung bình trượt 7 ngày và tìm 3 tuần có doanh thu cao nhất.

In [ ]:
# Viết code của bạn ở đây


### Bài C3. Đóng gói quy trình làm sạch

Viết hàm `lam_sach(duong_dan_raw, duong_dan_kh)` gom toàn bộ các bước A2–A6 thành một hàm, trả về DataFrame sạch.
Kiểm tra: kết quả của hàm phải có cùng số dòng với `df` ở Phần A. Vì sao đóng gói thành hàm lại quan trọng
khi dữ liệu được cập nhật hằng tuần?

In [ ]:
# Viết code của bạn ở đây
